In [3]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "source": ["# Notebook 10 — Hybrid Model 3: Entropy-Guided Stochastic Rebalancing\n",
    "Combines: (1) class-cap enforcement, (2) entropy maximisation, (3) seeded stochastic selection, (4) global distribution lock.\n",
    "Sensitive attribute: `fitness_level` only."]
  },
  {
   "cell_type": "code",
   "source": [
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib\n",
    "matplotlib.use('Agg')\n",
    "import matplotlib.pyplot as plt\n",
    "import json, os\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "os.makedirs('../data', exist_ok=True)\n",
    "os.makedirs('../results', exist_ok=True)\n",
    "\n",
    "df          = pd.read_csv('../data/t_close_anonymized.csv')\n",
    "df_original = pd.read_csv('../data/raw_fitness_data.csv')\n",
    "\n",
    "quasi_identifiers   = ['age', 'region', 'activity']\n",
    "sensitive_attribute = 'fitness_level'\n",
    "fitness_classes     = ['fit', 'moderately_fit', 'unfit']\n",
    "t_threshold         = 0.25\n",
    "CLASS_CAP           = 0.55   # no class allowed > 55% in any group\n",
    "ENTROPY_TARGET      = 0.95   # target normalised entropy\n",
    "SEED                = 42\n",
    "\n",
    "assert 'fitness_level' in df.columns, 'Run Notebooks 1-4 first.'\n",
    "print(f'Data loaded — shape: {df.shape}')\n",
    "print('fitness_level distribution:')\n",
    "print(df['fitness_level'].value_counts())"
   ]
  },
  {
   "cell_type": "code",
   "source": [
    "# Population reference\n",
    "pop_fl_dist = df_original['fitness_level'].value_counts(normalize=True).reindex(fitness_classes, fill_value=0)\n",
    "\n",
    "def tvd(p, q):\n",
    "    keys = q.index\n",
    "    p_a = p.reindex(keys, fill_value=0)\n",
    "    return 0.5 * sum(abs(p_a[i] - q[i]) for i in keys)\n",
    "\n",
    "def normalised_entropy(dist):\n",
    "    \"\"\"Shannon entropy normalised to [0,1] for 3-class distribution.\"\"\"\n",
    "    probs = np.array([dist.get(c, 0) for c in fitness_classes])\n",
    "    probs = probs[probs > 0]\n",
    "    if len(probs) == 0: return 0.0\n",
    "    H = -np.sum(probs * np.log(probs))\n",
    "    return H / np.log(len(fitness_classes))   # normalise by log(3)\n",
    "\n",
    "def inference_attack_accuracy(df_in, quasi_ids):\n",
    "    correct = 0\n",
    "    for _, gd in df_in.groupby(quasi_ids):\n",
    "        predicted = gd['fitness_level'].mode().iloc[0]\n",
    "        correct  += (gd['fitness_level'] == predicted).sum()\n",
    "    return correct / len(df_in)\n",
    "\n",
    "acc_before = inference_attack_accuracy(df, quasi_identifiers)\n",
    "print(f'Baseline inference attack accuracy: {acc_before:.4f} ({100*acc_before:.1f}%)')"
   ]
  },
  {
   "cell_type": "code",
   "source": [
    "# ---------------------------------------------------------------\n",
    "# HYBRID 3 — Entropy-Guided Stochastic Rebalancing\n",
    "#\n",
    "# For each equivalence group:\n",
    "#   Pass 1 — Class-cap enforcement (hard rule: no class > CLASS_CAP)\n",
    "#             Correct as many records as needed, not just 1\n",
    "#   Pass 2 — Entropy boost (if normalised entropy < ENTROPY_TARGET)\n",
    "#             Use seeded weighted-random selection among top-N candidates\n",
    "#   Pass 3 — TVD correction (same as Hybrid 2 but after caps are applied)\n",
    "#\n",
    "# Global lock: final fitness_level distribution must stay within\n",
    "# ±5% of original proportions to preserve utility.\n",
    "# ---------------------------------------------------------------\n",
    "\n",
    "rng = np.random.default_rng(SEED)   # seeded RNG — reproducible\n",
    "\n",
    "df_h3          = df.copy()\n",
    "total_adj      = 0\n",
    "pass1_adj      = 0\n",
    "pass2_adj      = 0\n",
    "pass3_adj      = 0\n",
    "\n",
    "for gk, gd in df_h3.groupby(quasi_identifiers):\n",
    "    if len(gd) < 3:\n",
    "        continue\n",
    "\n",
    "    # ---- PASS 1: Class-cap enforcement ----\n",
    "    for _ in range(5):   # iterate until all caps satisfied\n",
    "        curr   = df_h3.loc[gd.index, 'fitness_level']\n",
    "        dist   = curr.value_counts(normalize=True).reindex(fitness_classes, fill_value=0)\n",
    "        capped = dist[dist > CLASS_CAP]\n",
    "        if capped.empty:\n",
    "            break\n",
    "\n",
    "        over_c  = capped.idxmax()\n",
    "        under_c = (pop_fl_dist - dist).idxmax()  # pull towards population\n",
    "\n",
    "        over_records = df_h3.loc[gd.index][df_h3.loc[gd.index, 'fitness_level'] == over_c]\n",
    "        # How many to reassign: bring over_c to CLASS_CAP\n",
    "        n_excess = int(np.ceil((dist[over_c] - CLASS_CAP) * len(gd)))\n",
    "        n_excess = min(n_excess, len(over_records) - 1)\n",
    "        if n_excess <= 0:\n",
    "            break\n",
    "\n",
    "        # Seeded stochastic: choose among top-2*n candidates with step-based weights\n",
    "        candidates = over_records.nlargest(min(2 * n_excess + 2, len(over_records)), 'steps') \\\n",
    "                     if under_c == 'fit' else \\\n",
    "                     over_records.nsmallest(min(2 * n_excess + 2, len(over_records)), 'steps')\n",
    "        if under_c == 'moderately_fit':\n",
    "            med = over_records['steps'].median()\n",
    "            candidates = over_records.assign(_d=(over_records['steps'] - med).abs()).nsmallest(min(2*n_excess+2, len(over_records)), '_d')\n",
    "\n",
    "        # Weighted probability inversely proportional to deviation from uniform\n",
    "        weights = 1.0 / (np.arange(1, len(candidates) + 1, dtype=float))\n",
    "        weights /= weights.sum()\n",
    "        chosen_idx = rng.choice(len(candidates), size=min(n_excess, len(candidates)), replace=False, p=weights)\n",
    "        targets = candidates.index[chosen_idx]\n",
    "\n",
    "        df_h3.loc[targets, 'fitness_level'] = under_c\n",
    "        pass1_adj += len(targets)\n",
    "\n",
    "    # ---- PASS 2: Entropy boost ----\n",
    "    curr   = df_h3.loc[gd.index, 'fitness_level']\n",
    "    dist2  = curr.value_counts(normalize=True).reindex(fitness_classes, fill_value=0)\n",
    "    ent    = normalised_entropy(dist2)\n",
    "\n",
    "    if ent < ENTROPY_TARGET and len(gd) >= 6:\n",
    "        diff       = pop_fl_dist - dist2\n",
    "        under_c2   = diff.idxmax()   # most under-represented vs population\n",
    "        over_c2    = diff.idxmin()\n",
    "\n",
    "        over_recs2 = df_h3.loc[gd.index][df_h3.loc[gd.index, 'fitness_level'] == over_c2]\n",
    "        if len(over_recs2) >= 2:\n",
    "            # Seeded: pick 1 record with weighted random\n",
    "            n_cands = min(4, len(over_recs2))\n",
    "            if under_c2 == 'fit':\n",
    "                cands2 = over_recs2.nlargest(n_cands, 'steps')\n",
    "            elif under_c2 == 'unfit':\n",
    "                cands2 = over_recs2.nsmallest(n_cands, 'steps')\n",
    "            else:\n",
    "                med2 = over_recs2['steps'].median()\n",
    "                cands2 = over_recs2.assign(_d=(over_recs2['steps']-med2).abs()).nsmallest(n_cands,'_d')\n",
    "\n",
    "            w2 = 1.0 / (np.arange(1, len(cands2)+1, dtype=float))\n",
    "            w2 /= w2.sum()\n",
    "            ch2 = rng.choice(len(cands2), size=1, p=w2)\n",
    "            t2  = cands2.index[ch2]\n",
    "            df_h3.loc[t2, 'fitness_level'] = under_c2\n",
    "            pass2_adj += 1\n",
    "\n",
    "    # ---- PASS 3: TVD correction (residual) ----\n",
    "    curr3  = df_h3.loc[gd.index, 'fitness_level']\n",
    "    dist3  = curr3.value_counts(normalize=True).reindex(fitness_classes, fill_value=0)\n",
    "    if tvd(dist3, pop_fl_dist) > t_threshold:\n",
    "        diff3     = pop_fl_dist - dist3\n",
    "        under_c3  = diff3.idxmax()\n",
    "        over_c3   = diff3.idxmin()\n",
    "        over_r3   = df_h3.loc[gd.index][df_h3.loc[gd.index, 'fitness_level'] == over_c3]\n",
    "        if len(over_r3) >= 2:\n",
    "            if under_c3 == 'fit':   t3 = over_r3.nlargest(1,'steps').index\n",
    "            elif under_c3 == 'unfit': t3 = over_r3.nsmallest(1,'steps').index\n",
    "            else:\n",
    "                med3 = over_r3['steps'].median()\n",
    "                t3 = (over_r3['steps']-med3).abs().nsmallest(1).index\n",
    "            df_h3.loc[t3, 'fitness_level'] = under_c3\n",
    "            pass3_adj += 1\n",
    "\n",
    "total_adj = pass1_adj + pass2_adj + pass3_adj\n",
    "print(f'✅ Hybrid 3 complete')\n",
    "print(f'   Pass 1 (class-cap):  {pass1_adj} adjustments')\n",
    "print(f'   Pass 2 (entropy):    {pass2_adj} adjustments')\n",
    "print(f'   Pass 3 (TVD fix):    {pass3_adj} adjustments')\n",
    "print(f'   Total adjustments:   {total_adj}')\n",
    "print('\\nfitness_level distribution after Hybrid 3:')\n",
    "print(df_h3['fitness_level'].value_counts())"
   ]
  },
  {
   "cell_type": "code",
   "source": [
    "# ---- Global distribution lock check ----\n",
    "orig_dist  = df_original['fitness_level'].value_counts(normalize=True)\n",
    "final_dist = df_h3['fitness_level'].value_counts(normalize=True)\n",
    "\n",
    "print('Global distribution drift check (must stay within ±5%):')\n",
    "for cls in fitness_classes:\n",
    "    orig_p  = orig_dist.get(cls, 0)\n",
    "    final_p = final_dist.get(cls, 0)\n",
    "    drift   = abs(final_p - orig_p) * 100\n",
    "    status  = '✅' if drift <= 5 else '⚠️'\n",
    "    print(f'  {cls}: original={orig_p:.3f}, after={final_p:.3f}, drift={drift:.2f}% {status}')"
   ]
  },
  {
   "cell_type": "code",
   "source": [
    "# ---- Post-correction metrics ----\n",
    "viol_after = 0\n",
    "tvd_after  = []\n",
    "\n",
    "for _, gd in df_h3.groupby(quasi_identifiers):\n",
    "    gd_dist = gd['fitness_level'].value_counts(normalize=True).reindex(fitness_classes, fill_value=0)\n",
    "    d = tvd(gd_dist, pop_fl_dist)\n",
    "    tvd_after.append(d)\n",
    "    if d > t_threshold:\n",
    "        viol_after += 1\n",
    "\n",
    "acc_after = inference_attack_accuracy(df_h3, quasi_identifiers)\n",
    "\n",
    "print(f'Inference attack accuracy BEFORE Hybrid 3: {acc_before:.4f} ({100*acc_before:.1f}%)')\n",
    "print(f'Inference attack accuracy AFTER  Hybrid 3: {acc_after:.4f} ({100*acc_after:.1f}%)')\n",
    "print(f'Reduction: {100*(acc_before - acc_after):.1f} percentage points')\n",
    "print(f'Violations remaining: {viol_after} / {len(tvd_after)}')\n",
    "print(f'Mean TVD: {np.mean(tvd_after):.4f}')"
   ]
  },
  {
   "cell_type": "code",
   "source": [
    "# ---- Visualisations ----\n",
    "fig, axes = plt.subplots(1, 3, figsize=(15, 4))\n",
    "\n",
    "# TVD before/after\n",
    "tvd_before = [tvd(gd['fitness_level'].value_counts(normalize=True).reindex(fitness_classes,fill_value=0), pop_fl_dist)\n",
    "              for _, gd in df.groupby(quasi_identifiers)]\n",
    "axes[0].hist(tvd_before, bins=20, alpha=0.6, color='#e74c3c', label='Before')\n",
    "axes[0].hist(tvd_after,  bins=20, alpha=0.6, color='#2ecc71', label='After')\n",
    "axes[0].axvline(x=t_threshold, color='navy', linestyle='--', label=f't={t_threshold}')\n",
    "axes[0].set_title('TVD Distribution'); axes[0].set_xlabel('TVD'); axes[0].legend()\n",
    "\n",
    "# fitness_level counts\n",
    "x = np.arange(3); w = 0.35\n",
    "bc = df['fitness_level'].value_counts().reindex(fitness_classes).values\n",
    "ac = df_h3['fitness_level'].value_counts().reindex(fitness_classes).values\n",
    "axes[1].bar(x-w/2, bc, w, label='Before', color='#e74c3c', alpha=0.8)\n",
    "axes[1].bar(x+w/2, ac, w, label='After',  color='#2ecc71', alpha=0.8)\n",
    "axes[1].set_xticks(x); axes[1].set_xticklabels(fitness_classes, rotation=12)\n",
    "axes[1].set_title('fitness_level Class Counts'); axes[1].legend()\n",
    "\n",
    "# Inference attack accuracy comparison\n",
    "axes[2].bar(['Before H3','After H3'], [acc_before, acc_after], color=['#e74c3c','#2ecc71'])\n",
    "axes[2].set_ylim(0,1); axes[2].set_title('Inference Attack Accuracy')\n",
    "for i,v in enumerate([acc_before,acc_after]):\n",
    "    axes[2].text(i, v+0.01, f'{v:.3f}', ha='center')\n",
    "\n",
    "plt.suptitle('Hybrid 3 — Entropy-Guided Stochastic Rebalancing (fitness_level)', fontweight='bold')\n",
    "plt.tight_layout()\n",
    "plt.savefig('../results/hybrid3_analysis.png', dpi=120, bbox_inches='tight')\n",
    "plt.close()\n",
    "print('Saved: ../results/hybrid3_analysis.png')"
   ]
  },
  {
   "cell_type": "code",
   "source": [
    "df_h3.to_csv('../data/hybrid3_result.csv', index=False)\n",
    "print('Saved: ../data/hybrid3_result.csv')\n",
    "\n",
    "h3_metrics = {\n",
    "    'method': 'Hybrid 3 — Entropy-Guided Stochastic Rebalancing',\n",
    "    'sensitive_attribute': 'fitness_level',\n",
    "    'inference_attack_accuracy': float(acc_after),\n",
    "    'inference_attack_accuracy_before': float(acc_before),\n",
    "    'attack_accuracy_reduction': float(acc_before - acc_after),\n",
    "    'violations_after': viol_after,\n",
    "    'mean_tvd': float(np.mean(tvd_after)),\n",
    "    'total_adjustments': total_adj,\n",
    "    'pass1_class_cap': pass1_adj,\n",
    "    'pass2_entropy': pass2_adj,\n",
    "    'pass3_tvd': pass3_adj,\n",
    "    'data_loss_percent': 0.0\n",
    "}\n",
    "\n",
    "with open('../results/hybrid3_metrics.json', 'w') as f:\n",
    "    json.dump(h3_metrics, f, indent=2)\n",
    "\n",
    "print('\\nHybrid 3 summary:')\n",
    "for k, v in h3_metrics.items():\n",
    "    print(f'  {k}: {v}')"
   ]
  }
 ]
}

{'cells': [{'cell_type': 'markdown',
   'source': ['# Notebook 10 — Hybrid Model 3: Entropy-Guided Stochastic Rebalancing\n',
    'Combines: (1) class-cap enforcement, (2) entropy maximisation, (3) seeded stochastic selection, (4) global distribution lock.\n',
    'Sensitive attribute: `fitness_level` only.']},
  {'cell_type': 'code',
   'source': ['import pandas as pd\n',
    'import numpy as np\n',
    'import matplotlib\n',
    "matplotlib.use('Agg')\n",
    'import matplotlib.pyplot as plt\n',
    'import json, os\n',
    'import warnings\n',
    "warnings.filterwarnings('ignore')\n",
    '\n',
    "os.makedirs('../data', exist_ok=True)\n",
    "os.makedirs('../results', exist_ok=True)\n",
    '\n',
    "df          = pd.read_csv('../data/t_close_anonymized.csv')\n",
    "df_original = pd.read_csv('../data/raw_fitness_data.csv')\n",
    '\n',
    "quasi_identifiers   = ['age', 'region', 'activity']\n",
    "sensitive_attribute = 'fitness_level'\n",
    "fitness_classes     = ['fit'